# Notebook 06 — Failure Analysis

**Objective.** Label failures by taxonomy, extract representative cases, build Blog-1 panels.

**Inputs.** Raw predictions + references.

**Expected outputs.** results/failures/failure_cases.csv, qualitative panels.

_Companion notebook: runnable top-to-bottom, relative paths, no hidden state, saves artifacts to `results/`._

In [1]:
import sys, os
# Reusable logic lives in src/grounded_memory/. Notebooks call it.
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
from pathlib import Path
RESULTS = Path('..') / 'results'
from grounded_memory import __version__
print('grounded_memory', __version__)

grounded_memory 0.1.0


## Failure taxonomy

Categories: object/attribute hallucination, relation/spatial error, action/intent inference, overconfidence, failure to abstain, evidence mismatch, right-for-wrong-reason.

In [2]:
categories = ['object_hallucination','attribute_hallucination','relation_error',
  'spatial_error','action_inference','identity_assumption','intent_inference',
  'unsupported_causal','overconfidence','failure_to_abstain','evidence_mismatch',
  'right_for_wrong_reason']
print(len(categories), 'failure categories')

12 failure categories


## Select a stratified sample + build the rating queue

48 real images (8 per manifest bucket, seed `20260815` -- this project's
standard seed, also used in `configs/datasets.yaml`/`configs/evaluation.yaml`)
x both real E1 models x all 3 prompt conditions = **288 items** to rate. The
constants below are the only things you need to edit to change scope (e.g.
narrow `MODEL_NAMES` to one model, or lower `PER_BUCKET`, for a smaller pass).

In [3]:
import json
import textwrap
from collections import defaultdict
from datetime import UTC, datetime

from grounded_memory.config import load_config
from grounded_memory.data.gqa import load_scene_graphs, scene_graph_path
from grounded_memory.data.human_eval import RatingRow, load_ratings, save_ratings
from grounded_memory.data.review import draw_boxes, load_manifest, scene_graph_to_triples
from grounded_memory.data.selection import stratified_sample
from grounded_memory.evaluation.structured_scene import score_structured_scene
from grounded_memory.scene.schema import SceneRepresentation

PER_BUCKET = 8
SEED = 20260815
MODEL_NAMES: list[str] | None = None  # None = both real E1 models found in the frozen predictions
CONDITIONS = ['standard', 'grounded', 'structured']
RATER = 'sanja'  # edit before rating
FAILURE_CASES_CSV = RESULTS / 'failures' / 'failure_cases.csv'

manifest_rows = load_manifest(Path('..') / 'data' / 'manifests' / 'gqa_benchmark_v1.csv')
sample_rows = stratified_sample(manifest_rows, per_bucket=PER_BUCKET, seed=SEED)
print(f'Sampled {len(sample_rows)} images ({PER_BUCKET}/bucket, seed={SEED}).')

Sampled 48 images (8/bucket, seed=20260815).


In [4]:
def load_jsonl(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text().splitlines() if line]

predictions_dir = RESULTS / 'predictions'
files = sorted(predictions_dir.glob('e1_*.jsonl'))  # top-level only -- frozen 720-record run
records_by_key: dict[tuple[str, str, str], dict] = {}
for f in files:
    for rec in load_jsonl(f):
        condition = rec.get('metrics', {}).get('condition', '?')
        records_by_key[(rec['model_id'], condition, rec['sample_id'])] = rec

model_names = MODEL_NAMES or sorted({k[0] for k in records_by_key})
print(f'{len(records_by_key)} predictions indexed. Models: {model_names}')

# gqa.root is repo-root-relative -- resolve it against the repo root explicitly,
# since the notebook kernel's cwd is notebooks/, not the repo root (same fix
# as notebooks 02/03/04/05).
data_cfg = load_config('datasets')
REPO_ROOT = Path('..').resolve()
gqa_root = Path(data_cfg['gqa']['root'])
if not gqa_root.is_absolute():
    gqa_root = (REPO_ROOT / gqa_root).resolve()
data_cfg['gqa']['root'] = str(gqa_root)
GQA_IMAGES_DIR = gqa_root / data_cfg['gqa']['images_dir']
scene_graphs = load_scene_graphs(scene_graph_path(data_cfg))
print(f'{len(scene_graphs)} scene graphs loaded.')

720 predictions indexed. Models: ['ollama/llava:7b', 'ollama/qwen3-vl:8b']


10696 scene graphs loaded.

In [5]:
# Resume from a prior session's failure_cases.csv if it exists -- already-rated
# items keep their ratings instead of being reset (same resume pattern as
# notebook 02's REVIEW_CSV).
existing_by_key: dict[tuple[str, str, str], RatingRow] = {}
if FAILURE_CASES_CSV.exists():
    for r in load_ratings(FAILURE_CASES_CSV):
        existing_by_key[(r.image_id, r.model_id, r.condition)] = r
    print(f'Resuming: {len(existing_by_key)} existing rating rows found in {FAILURE_CASES_CSV.name}')
else:
    print(f'No existing {FAILURE_CASES_CSV.name} -- starting fresh (file is created on first Save).')

queue: list[RatingRow] = []
missing = 0
for row in sample_rows:
    for model_id in model_names:
        for condition in CONDITIONS:
            key = (model_id, condition, row.image_id)
            rec = records_by_key.get(key)
            if rec is None:
                missing += 1
                continue
            existing_row = existing_by_key.get((row.image_id, model_id, condition))
            if existing_row is not None:
                queue.append(existing_row)
                continue
            excerpt = textwrap.shorten(
                rec.get('prediction', '').replace('\n', ' '), width=800, placeholder=' …'
            )
            queue.append(RatingRow(
                image_id=row.image_id, primary_bucket=row.primary_bucket,
                model_id=model_id, condition=condition, prediction_excerpt=excerpt,
            ))

n_rated = sum(1 for r in queue if r.rated_at)
print(f'{len(queue)} items queued ({missing} missing predictions -- unexpected if the manifest '
      f'sample and the frozen E1 run are both intact).')
print(f'{n_rated}/{len(queue)} already rated.' if n_rated else 'Starting fresh -- nothing rated yet.')

No existing failure_cases.csv -- starting fresh (file is created on first Save).
288 items queued (0 missing predictions -- unexpected if the manifest sample and the frozen E1 run are both intact).
Starting fresh -- nothing rated yet.


## Rate each item

Each screen shows: the image with GQA boxes + labels, the GQA ground-truth
triples (reference, not omniscient truth -- an unannotated claim isn't
automatically wrong), the model's full raw prediction, and -- for
structured-condition items with a valid JSON parse -- an automatic hint line
(evidence-support rate, object recall) from the same scoring path notebook 05
uses. That hint is context, not a substitute for your judgment.

Rate on 4 binary items:

- **Object/Attribute Hallucination** -- Yes = a hallucination was observed (bad).
- **Relation/Spatial Error** -- Yes = an error was observed (bad).
- **Unsupported Action/Intent Inference** -- Yes = an unsupported inference was observed (bad).
- **Appropriate Abstention/Hedging** -- Yes = the response correctly hedged/abstained (**good** --
  opposite polarity from the other three items above).

Plus an optional failure-taxonomy tag (from the 12 categories above, for any
item marked Yes) and free-text notes. Progress saves to
`results/failures/failure_cases.csv` on every Save click (full rewrite, safe
to stop and resume later).

In [6]:
def render_item(idx: int) -> None:
    import matplotlib.pyplot as plt
    from PIL import Image

    r = queue[idx]
    rec = records_by_key.get((r.model_id, r.condition, r.image_id))
    sg = scene_graphs.get(r.image_id)
    image_path = GQA_IMAGES_DIR / f'{r.image_id}.jpg'

    print(f'--- item {idx + 1} / {len(queue)}   image_id={r.image_id}   '
          f'bucket={r.primary_bucket} ---')
    print(f'model={r.model_id}   condition={r.condition}')

    fig, ax = plt.subplots(figsize=(6, 6))
    if image_path.exists() and sg is not None:
        draw_boxes(image_path, sg, ax=ax)
    elif image_path.exists():
        ax.imshow(Image.open(image_path))
        ax.set_axis_off()
    else:
        ax.set_axis_off()
        ax.text(0.5, 0.5, f'image not found:\n{image_path}', ha='center', va='center', wrap=True)
    plt.show()

    print('\nGQA ground-truth reference (incomplete -- not omniscient):')
    if sg is not None:
        for line in scene_graph_to_triples(sg):
            print(f'  {line}')
    else:
        print('  (no scene graph loaded)')

    print(f'\n--- {r.model_id} / {r.condition} raw prediction ---')
    print(rec.get('prediction', '(missing)') if rec else '(no matching prediction record)')

    if r.condition == 'structured' and rec and rec.get('parse_ok') and rec.get('parsed_prediction') and sg is not None:
        scene = SceneRepresentation.model_validate(rec['parsed_prediction'])
        score = score_structured_scene(scene, sg)
        print(f'\n[automatic hint, not a substitute for your judgment] '
              f'evidence_support_rate={score.evidence_support_rate:.2f}   '
              f'object_recall={score.objects.recall:.2f}')


def _progress_text() -> str:
    return f'{sum(1 for x in queue if x.rated_at)} / {len(queue)} rated so far'

In [7]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    _idx = 0

    oah_cb = widgets.Checkbox(description='Object/Attribute Hallucination')
    rse_cb = widgets.Checkbox(description='Relation/Spatial Error')
    uaii_cb = widgets.Checkbox(description='Unsupported Action/Intent Inference')
    aah_cb = widgets.Checkbox(description='Appropriate Abstention/Hedging (Yes = GOOD)')
    taxonomy_dropdown = widgets.Dropdown(options=['n/a'] + categories, description='taxonomy:')
    notes_box = widgets.Textarea(description='notes:', layout=widgets.Layout(width='500px'))
    rater_box = widgets.Text(value=RATER, description='rater:')
    prev_btn = widgets.Button(description='< prev')
    next_btn = widgets.Button(description='next >')
    save_btn = widgets.Button(description='Save', button_style='success')
    position_label = widgets.Label()
    out = widgets.Output()

    def _load_into_widgets(idx):
        r = queue[idx]
        oah_cb.value = r.object_attribute_hallucination
        rse_cb.value = r.relation_spatial_error
        uaii_cb.value = r.unsupported_action_intent_inference
        aah_cb.value = r.appropriate_abstention_hedging
        taxonomy_dropdown.value = r.failure_taxonomy or 'n/a'
        notes_box.value = r.notes
        if r.rater:
            rater_box.value = r.rater
        position_label.value = f'{idx + 1} / {len(queue)}   ({_progress_text()})'
        out.clear_output(wait=True)
        with out:
            render_item(idx)

    def _on_prev(_):
        global _idx
        _idx = (_idx - 1) % len(queue)
        _load_into_widgets(_idx)

    def _on_next(_):
        global _idx
        _idx = (_idx + 1) % len(queue)
        _load_into_widgets(_idx)

    def _on_save(_):
        r = queue[_idx]
        r.object_attribute_hallucination = oah_cb.value
        r.relation_spatial_error = rse_cb.value
        r.unsupported_action_intent_inference = uaii_cb.value
        r.appropriate_abstention_hedging = aah_cb.value
        r.failure_taxonomy = '' if taxonomy_dropdown.value == 'n/a' else taxonomy_dropdown.value
        r.notes = notes_box.value
        r.rater = rater_box.value
        r.rated_at = datetime.now(UTC).isoformat(timespec='seconds')
        save_ratings(queue, FAILURE_CASES_CSV)
        position_label.value = f'{_idx + 1} / {len(queue)}   ({_progress_text()})'
        with out:
            print(f'Saved item {_idx} -> {FAILURE_CASES_CSV.name}')

    prev_btn.on_click(_on_prev)
    next_btn.on_click(_on_next)
    save_btn.on_click(_on_save)

    display(widgets.HBox([prev_btn, next_btn, save_btn, position_label]))
    display(widgets.HBox([oah_cb, rse_cb]))
    display(widgets.HBox([uaii_cb, aah_cb]))
    display(taxonomy_dropdown, notes_box, rater_box, out)
    _load_into_widgets(_idx)

except ImportError:
    print('ipywidgets not available -- use rate_item_fallback(idx, ...) below instead.')

Dropdown(description='taxonomy:', options=('n/a', 'object_hallucination', 'attribute_hallucination', 'relation…

Textarea(value='', description='notes:', layout=Layout(width='500px'))

Text(value='sanja', description='rater:')

Output()

In [8]:
def rate_item_fallback(
    idx: int,
    object_attribute_hallucination: bool | None = None,
    relation_spatial_error: bool | None = None,
    unsupported_action_intent_inference: bool | None = None,
    appropriate_abstention_hedging: bool | None = None,
    failure_taxonomy: str | None = None,
    notes: str | None = None,
    rater: str | None = None,
) -> None:
    """Plain-function rating path for when ipywidgets isn't installed.

    Call with just `idx` to view the item; pass any rubric kwargs to record +
    save. Example:
        rate_item_fallback(0)                                         # just look
        rate_item_fallback(0, object_attribute_hallucination=True,
                            failure_taxonomy='object_hallucination',
                            rater='sanja')                              # rate + save
    """
    r = queue[idx]
    changed = any(v is not None for v in (
        object_attribute_hallucination, relation_spatial_error,
        unsupported_action_intent_inference, appropriate_abstention_hedging,
        failure_taxonomy, notes, rater,
    ))
    if object_attribute_hallucination is not None:
        r.object_attribute_hallucination = object_attribute_hallucination
    if relation_spatial_error is not None:
        r.relation_spatial_error = relation_spatial_error
    if unsupported_action_intent_inference is not None:
        r.unsupported_action_intent_inference = unsupported_action_intent_inference
    if appropriate_abstention_hedging is not None:
        r.appropriate_abstention_hedging = appropriate_abstention_hedging
    if failure_taxonomy is not None:
        r.failure_taxonomy = failure_taxonomy
    if notes is not None:
        r.notes = notes
    if rater is not None:
        r.rater = rater
    if changed:
        r.rated_at = datetime.now(UTC).isoformat(timespec='seconds')
        save_ratings(queue, FAILURE_CASES_CSV)
        print(f'Saved item {idx} -> {FAILURE_CASES_CSV.name}')
    render_item(idx)
    print(f'\n{_progress_text()}')

## Rating progress summary

Descriptive only -- reads whatever's currently in `failure_cases.csv` (via
`queue`, which was seeded from it above). Safe to run at any point, including
before any rating has happened.

In [9]:
import pandas as pd

rated = [r for r in queue if r.rated_at]
print(f'{len(rated)} / {len(queue)} items rated.')

RUBRIC_COLUMNS = [
    'object_attribute_hallucination', 'relation_spatial_error',
    'unsupported_action_intent_inference', 'appropriate_abstention_hedging',
]
if rated:
    df = pd.DataFrame([r.__dict__ for r in rated])
    for col in RUBRIC_COLUMNS:
        print(f'\n{col} = True rate by (model_id, condition):')
        print(df.groupby(['model_id', 'condition'])[col].mean())
else:
    print('Nothing rated yet -- open this notebook in a live kernel and use the widgets above.')

0 / 288 items rated.
Nothing rated yet -- open this notebook in a live kernel and use the widgets above.


## Findings

_Fill in 3-5 concrete findings after **completing** the human rating pass
above -- this requires actually clicking through the 288 items in a live
Jupyter/VS Code kernel; it cannot be generated by re-running this notebook
headlessly, and these numbers must not be fabricated._

1. …
2. …
3. …

> Do not populate tables with invented numbers. If the full model/data is unavailable, leave clearly-labeled placeholders.